# CEC Transmission Line Density: Weather Grid Mapping

**Purpose:** Map the CEC transmission line density NetCDF
(`transmission_line_density.CA.nc`) onto the weather reanalysis grid
using a KD-Tree nearest-neighbour lookup, producing a parquet file
for use in `02_06_Add_OpenStreetMap_to_Static_Feature.ipynb`.

**Pipeline Overview:**

| Phase | Steps |
|-------|-------|
| **A. Inspect** | Load NC file; check dimensions, variable name, missing rate |
| **B. Build Reference Grid** | Extract (lon, lat) pairs from the weather grid |
| **C. KD-Tree Match** | Nearest-neighbour match; exact geodesic distances |
| **D. Assign & Save** | Apply 4 km distance mask; fill unmatched with 0; save |

**Input:** `Power_Data/transmission_line_density.CA.nc`

**Reference grid:** `Clean_Data/Veg_Data/lon_lat_pair_weather_match_veg_v2.parquet`

**Output:** `Clean_Data/Power_Data/transmission_line_density_match_weather_grid.parquet`

> **Run order:** This notebook must run **before `02_06`**.

## 0. Configuration

Centralized path configuration — **edit this cell only**.

In [1]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

# Input: CEC transmission line density NC file
NC_PATH = os.path.join(PROJECT_ROOT, "Power_Data",
                       "transmission_line_density.CA.nc")

# Reference grid: veg-filtered weather grid (lon/lat pairs)
REFERENCE_GRID_PATH = os.path.join(
    PROJECT_ROOT, "Clean_Data", "Veg_Data",
    "lon_lat_pair_weather_match_veg_v2.parquet"
)

# Distance threshold — cells > this km from nearest data pixel set to 0
MAX_MATCH_KM = 4.0

# Output
OUTPUT_DIR  = os.path.join(PROJECT_ROOT, "Clean_Data", "Power_Data")
OUTPUT_PATH = os.path.join(OUTPUT_DIR,
                           "transmission_line_density_match_weather_grid.parquet")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"NC file        : {NC_PATH}")
print(f"Reference grid : {REFERENCE_GRID_PATH}")
print(f"Output         : {OUTPUT_PATH}")
for label, path in [("NC_PATH",            NC_PATH),
                    ("REFERENCE_GRID_PATH", REFERENCE_GRID_PATH)]:
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {label}")

NC file        : E:\zcao\CA_Wildfire\Power_Data\transmission_line_density.CA.nc
Reference grid : E:\zcao\CA_Wildfire\Clean_Data\Veg_Data\lon_lat_pair_weather_match_veg_v2.parquet
Output         : E:\zcao\CA_Wildfire\Clean_Data\Power_Data\transmission_line_density_match_weather_grid.parquet
  [OK] NC_PATH
  [OK] REFERENCE_GRID_PATH


## 1. Environment Setup

In [2]:
import sys, gc, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import xarray as xr
from scipy.spatial import cKDTree
from geopy.distance import geodesic
from tqdm import tqdm

gc.collect()
print(f"Python : {sys.version.split('|')[0].strip()}")
print(f"pandas : {pd.__version__}")
print(f"xarray : {xr.__version__}")
print(f"numpy  : {np.__version__}")

Python : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas : 2.2.2
xarray : 2024.7.0
numpy  : 1.24.4


---

# Phase A: Inspect NC File

Load and inspect the transmission line density NetCDF — dimensions,
variable name, spatial coverage, and missing rate.

## 2. Load & Inspect

In [3]:
ds = xr.open_dataset(NC_PATH)

print(f"Dimensions  : {dict(ds.dims)}")
print(f"Coordinates : {list(ds.coords)}")
print(f"Variables   : {list(ds.data_vars)}")
ds

Dimensions  : {'lat': 585, 'lon': 1386}
Coordinates : ['lat', 'lon']
Variables   : ['line_density_km_per_cell']


<xarray.Dataset> Size: 7MB
Dimensions:                   (lat: 585, lon: 1386)
Coordinates:
  * lat                       (lat) float64 5kB 25.07 25.11 25.15 ... 49.36 49.4
  * lon                       (lon) float64 11kB -124.8 -124.7 ... -67.1 -67.06
Data variables:
    line_density_km_per_cell  (lat, lon) float64 6MB ...

In [4]:
VAR_NAME = list(ds.data_vars)[0]  # auto-detect variable name
print(f"Variable name : '{VAR_NAME}'")
print(f"(will be saved as column: 'line_density_km_per_cell')")

density_df = ds.to_dataframe().reset_index()
ds.close()

print(f"\nShape (all rows)   : {density_df.shape}")
print(f"Missing rate       : {density_df[VAR_NAME].isna().mean():.3f}")
density_df = density_df.dropna(subset=[VAR_NAME])
print(f"Shape (non-NA)     : {density_df.shape}")
print(f"\nLon range : {density_df['lon'].min():.4f} → {density_df['lon'].max():.4f}")
print(f"Lat range : {density_df['lat'].min():.4f} → {density_df['lat'].max():.4f}")
print(f"Value range : {density_df[VAR_NAME].min():.4f} → {density_df[VAR_NAME].max():.4f}")

Variable name : 'line_density_km_per_cell'
(will be saved as column: 'line_density_km_per_cell')

Shape (all rows)   : (810810, 3)
Missing rate       : 0.000
Shape (non-NA)     : (810810, 3)

Lon range : -124.7667 → -67.0583
Lat range : 25.0667 → 49.4000
Value range : 0.0000 → 65.7393


---

# Phase B: Build Reference Weather Grid

Extract unique (lon, lat) pairs from the veg-filtered weather grid.
This defines the target spatial resolution for the mapping.

In [5]:
reference_grid = pd.read_parquet(REFERENCE_GRID_PATH)
reference_grid = reference_grid[['lon','lat']].drop_duplicates().reset_index(drop=True)

print(f"Reference grid cells : {len(reference_grid):,}")
print(f"Lon range : {reference_grid['lon'].min():.4f} → {reference_grid['lon'].max():.4f}")
print(f"Lat range : {reference_grid['lat'].min():.4f} → {reference_grid['lat'].max():.4f}")

Reference grid cells : 17,714
Lon range : -124.3917 → -115.9333
Lat range : 32.5250 → 42.0667


---

# Phase C: KD-Tree Nearest-Neighbour Match

Build a KD-Tree on the density data coordinates, find the nearest pixel
for each weather grid cell, then compute exact geodesic distances (km).
Cells more than `MAX_MATCH_KM` away are treated as no-data and set to 0.

## 4. Approximate Match (KD-Tree)

In [6]:
tree = cKDTree(density_df[['lat','lon']].values)
approx_dist, indices = tree.query(reference_grid[['lat','lon']].values, k=1)

print(f"Approximate distance range (degrees): "
      f"{approx_dist.min():.4f} → {approx_dist.max():.4f}")

Approximate distance range (degrees): 0.0000 → 0.0000


## 5. Exact Geodesic Distances

In [7]:
exact_dist   = np.empty(len(reference_grid))
grid_coords  = reference_grid[['lat','lon']].values
power_coords = density_df[['lat','lon']].values

for i in tqdm(range(len(reference_grid)), desc='Geodesic distances'):
    exact_dist[i] = geodesic(
        (grid_coords[i,0],       grid_coords[i,1]),
        (power_coords[indices[i],0], power_coords[indices[i],1])
    ).km

print(f"Exact distance range (km): {exact_dist.min():.4f} → {exact_dist.max():.4f}")
print(f"Cells within {MAX_MATCH_KM} km  : "
      f"{(exact_dist < MAX_MATCH_KM).sum():,} / {len(reference_grid):,} "
      f"({(exact_dist < MAX_MATCH_KM).mean()*100:.1f}%)")

Geodesic distances: 100%|██████████| 17714/17714 [00:01<00:00, 9967.39it/s] 

Exact distance range (km): 0.0000 → 0.0000
Cells within 4.0 km  : 17,714 / 17,714 (100.0%)


---

# Phase D: Assign Values & Save

Apply the distance mask and assign density values. Cells beyond
`MAX_MATCH_KM` receive 0 (no nearby infrastructure).

In [8]:
mask = exact_dist < MAX_MATCH_KM

reference_grid['line_density_km_per_cell'] = np.where(
    mask,
    density_df.iloc[indices][VAR_NAME].values,
    0.0
)

print(f"Matched cells ({MAX_MATCH_KM} km threshold) : {mask.sum():,}")
print(f"Unmatched → 0                              : {(~mask).sum():,}")
print(f"\nline_density_km_per_cell stats:")
print(reference_grid['line_density_km_per_cell'].describe().to_string())

Matched cells (4.0 km threshold) : 17,714
Unmatched → 0                              : 0

line_density_km_per_cell stats:
count    17714.000000
mean         2.579251
std          5.542844
min          0.000000
25%          0.000000
50%          0.000000
75%          3.638138
max         65.739266


In [9]:
output = reference_grid[['lon','lat','line_density_km_per_cell']]

output.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved -> {OUTPUT_PATH}")
print(f"File size  : {os.path.getsize(OUTPUT_PATH)/1e6:.1f} MB")
print(f"Final shape: {output.shape[0]:,} rows × {output.shape[1]} cols")
print(f"Columns    : {list(output.columns)}")

del density_df, reference_grid, tree, approx_dist, indices, exact_dist, mask
gc.collect()

Saved -> E:\zcao\CA_Wildfire\Clean_Data\Power_Data\transmission_line_density_match_weather_grid.parquet
File size  : 0.1 MB
Final shape: 17,714 rows × 3 cols
Columns    : ['lon', 'lat', 'line_density_km_per_cell']


30

## 6. Summary

| Phase | Step | Description | Key Result |
|-------|------|-------------|------------|
| A | Load NC | `transmission_line_density.CA.nc` | Auto-detect variable name |
| A | Inspect | Dimensions, missing rate, value range | Non-NA rows kept |
| B | Reference grid | Veg-filtered weather (lon, lat) pairs | ~13k grid cells |
| C | KD-Tree | Approximate nearest neighbour (degrees) | Fast initial match |
| C | Geodesic | Exact km distances for all matches | QA — distance range |
| D | Mask | Distance > 4.0 km → 0 | Ocean / boundary cells zeroed |
| D | Save | `transmission_line_density_match_weather_grid.parquet` | Input for `02_06` |